# Train MFFT-Base on Kaggle
**Multi-Frequency Fusion Transformer — Base Variant (1.62M params)**

## Setup
1. **Settings → Accelerator**: GPU T4 x2 (or P100)
2. **Settings → Internet**: On
3. **Add Input** (attach all 11 datasets):
   - `stable-diffusion`, `places365`, `open-images-v7-dataset`
   - `ntire2026`, `midjourney`, `mfft-real`, `genimage-ai`
   - `faceforensics`, `dfdc-faces-of-the-train-sample`
   - `dall-e3`, `celebdf-v2image-dataset`
4. **Secrets → Add Secret**: `HF_TOKEN` = your HuggingFace write token

## What happens
- **First run**: Bootstraps a frozen split manifest from mounted datasets → uploads to HF
- **Later runs**: Downloads the existing manifest from HF → same data as all other runs
- Checkpoints go to `runs/<run_id>/...` (no collisions between parallel sessions)

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo exists at {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))

subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv"], check=False)
print("Deps installed.")

In [ ]:
# Cell 2: Verify GPU & HF token
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "No GPU")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: HF_TOKEN not found as Kaggle Secret: {e}")

In [ ]:
# Cell 3: Run training (MFFT-Base)
# This calls SplitManifestManager → KaggleDatasetLoader → trains
%run /kaggle/working/mfft_repo/kaggle_train_resumable.py

## Results

After training completes:
- Checkpoints: `https://huggingface.co/MohsinElis/mfft-checkpoints/tree/main/runs/<run_id>`
- Manifest: `https://huggingface.co/MohsinElis/mfft-master-manifest/blob/main/manifest/split_manifest.json`
- Run `list_active_runs.py` locally to see status across all parallel runs